In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Load Welsh CEFR dataset from HuggingFace
ds = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]
df_main = ds.to_pandas()  

# Load your B2 JSON data
df_b2 = pd.read_json("files/b2_welsh.json")
df_b2["cefr_level"] = "B2"
df_b2 = df_b2.drop_duplicates(subset="text", keep="first")

# Combine both DataFrames
df_combined = pd.concat([df_main, df_b2], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset="text", keep="first")

# Convert back to HuggingFace Dataset
ds_merged = Dataset.from_pandas(df_combined)

In [9]:
ds_merged

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text', '__index_level_0__'],
    num_rows: 2020
})

In [10]:
CEFR_LEVELS = ["A1", "A2", "B2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in ds_merged["cefr_level"]])

In [11]:
model_name = "eurobert_cefr_english_only/final_model"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [12]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [13]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [14]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [16]:
best_f1 = 0.0
best_trainer = None
best_tokenizer = None

for fold, (train_idx, val_idx) in enumerate(skf.split(ds_merged, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = ds_merged.select(train_idx)
    ds_val = ds_merged.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS),trust_remote_code=True)

    args = TrainingArguments(
        output_dir=f"./eurobert_cefr_welsh_en_cy_b2/fold_{fold}",  
        num_train_epochs=3, 
        per_device_train_batch_size=2,              
        per_device_eval_batch_size=3,                
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,      
        optim="adamw_torch_fused",                   
        lr_scheduler_type="linear",                  
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in CEFR_LEVELS:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)

    all_results.append(row)


 Running Fold 1...


Map: 100%|██████████| 404/404 [00:00<00:00, 11360.89 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_4244\2574579616.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B2 Precision,B2 Recall,B2 F1
1,1.111700,0.983852,0.566832,0.508945,0.582702,0.566832,0.524164,0.921569,0.668246,0.560000,0.115702,0.191781,0.672727,0.569231,0.616667
2,1.067900,1.015561,0.547030,0.450701,0.383340,0.547030,0.540179,0.790850,0.641910,0.000000,0.000000,0.000000,0.555556,0.769231,0.645161
3,0.916500,0.897312,0.574257,0.551159,0.570272,0.574257,0.556034,0.843137,0.670130,0.416667,0.247934,0.310881,0.730000,0.561538,0.634783


c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



 Running Fold 2...


...

Map: 100%|██████████| 404/404 [00:00<00:00, 9332.12 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_4244\2574579616.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B2 Precision,B2 Recall,B2 F1
1,1.073300,0.940651,0.495050,0.484329,0.663369,0.495050,0.833333,0.230263,0.360825,0.353160,0.785124,0.487179,0.752688,0.534351,0.625000
2,0.816000,0.683673,0.715347,0.715263,0.716342,0.715347,0.815068,0.782895,0.798658,0.620690,0.595041,0.607595,0.690141,0.748092,0.717949
3,0.527900,0.587398,0.779703,0.781221,0.785569,0.779703,0.860000,0.848684,0.854305,0.661765,0.743802,0.700389,0.813559,0.732824,0.771084



 Running Fold 5...


Map: 100%|██████████| 404/404 [00:00<00:00, 19242.77 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_4244\2574579616.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B2 Precision,B2 Recall,B2 F1
1,1.090100,0.964861,0.527228,0.524836,0.547775,0.527228,0.704762,0.486842,0.575875,0.381356,0.371901,0.376569,0.519337,0.717557,0.602564
2,0.790700,0.809457,0.678218,0.673115,0.700389,0.678218,0.695652,0.842105,0.761905,0.571429,0.661157,0.613027,0.825000,0.503817,0.625592
3,0.521900,0.703765,0.712871,0.712424,0.712028,0.712871,0.792208,0.802632,0.797386,0.621849,0.611570,0.616667,0.702290,0.702290,0.702290


In [17]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_welsh_en_cy_b2/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [19]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)


In [20]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.570272  0.574257  0.551159  0.556034  0.843137  0.670130   
1        2        0.718580  0.725248  0.720501  0.805882  0.895425  0.848297   
2        3        0.628799  0.633663  0.629765  0.678363  0.763158  0.718266   
3        4        0.785569  0.779703  0.781221  0.860000  0.848684  0.854305   
4        5        0.712028  0.712871  0.712424  0.792208  0.802632  0.797386   
5  Average        0.683050  0.685149  0.679014  0.738497  0.830607  0.777677   

         A2                            B2                      
  Precision    Recall        F1 Precision    Recall        F1  
0  0.416667  0.247934  0.310881  0.730000  0.561538  0.634783  
1  0.609091  0.553719  0.580087  0.717742  0.684615  0.700787  
2  0.468468  0.426230  0.446352  0.721311  0.676923  0.698413  
3  0.661765  0.743802  0.700389  0.813559  0.732824  0.771084  
4  0.621849  0.611570  0.616667  0.702290  0.702290  0.702290  
5  0.555568  0.516651  0.530875  0.736981  0.671638  0.701471